# Prompting Techniques with Ollama

Prompting steers a pretrained LLM at inference time, with **no weight updates**. In this notebook you will:

1. See four techniques in action: **few-shot**, **chain-of-thought**, **self-consistency**, **RAG**
2. Practice them yourself in short exercises

| Technique | Idea |
|---|---|
| Zero/Few-shot | Give instructions, optionally with examples |
| Chain-of-Thought (CoT) | Ask for step-by-step reasoning before the answer |
| Self-Consistency | Sample several CoT chains, majority-vote the answer |
| RAG | Retrieve relevant documents and put them in the prompt |

## Setup
Run once (Colab with GPU runtime). It installs Ollama and pulls a small model.

In [1]:
%pip install -q ollama scikit-learn numpy
!sudo apt-get install -y -qq zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!nohup ollama serve > ollama.log 2>&1 &
import time; time.sleep(5)
!ollama pull gemma4:e2b

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 6.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 



In [2]:
import ollama, re
from collections import Counter

MODEL = "gemma4:e2b"

def generate(prompt, max_tokens=150, temperature=0.0):
    """Send a prompt to the model and return its text. temperature=0 is deterministic."""
    r = ollama.generate(model=MODEL, prompt=prompt, think=False,
                        options={"temperature": temperature, "num_predict": max_tokens})
    return r["response"].strip()

print(generate("Say hello in one short sentence."))

Hello!


## Demo 1: Zero-shot vs. Few-shot
Few-shot adds input→output examples so the model copies the format.

In [3]:
review = "The pacing was dreadful, but the soundtrack was gorgeous."
task = "Classify the review as Positive, Negative, or Mixed. Reply with one word."

zero_shot = f"{task}\n\nReview: {review}\nLabel:"

few_shot = f"""{task}

Review: A masterpiece of modern cinema.
Label: Positive

Review: I fell asleep halfway through.
Label: Negative

Review: Great acting, but the plot made no sense.
Label: Mixed

Review: {review}
Label:"""

print("Zero-shot:", generate(zero_shot, max_tokens=10))
print("Few-shot :", generate(few_shot, max_tokens=10))

Zero-shot: Mixed
Few-shot : Mixed


## Demo 2: Chain-of-Thought
Asking the model to reason step by step helps on multi-step problems.

In [4]:
question = ("A store had 50 shirts. It sold 18 on Monday, got a shipment of 30, "
            "then sold half of what it had. How many shirts are left?")   # answer: 31

print("Direct:\n", generate(f"Q: {question}\nGive only the final number.\nA:", max_tokens=10))
print("\nZero-shot CoT:\n", generate(f"Q: {question}\nA: Let's think step by step.", max_tokens=200))

Direct:
 36

Zero-shot CoT:
 Here is the step-by-step solution:

**1. Starting Inventory:**
The store begins with **50** shirts.

**2. Sales on Monday:**
They sold 18 shirts.
$50 - 18 = 32$ shirts remaining.

**3. Shipment Received:**
They received a shipment of 30 shirts.
$32 + 30 = 62$ shirts total.

**4. Second Sale:**
They sold half of what they had (half of 62).
Half of 62 is $62 / 2 = 31$ shirts sold.

**5. Final Count:**
Subtract the second sale from the current total:
$62 - 31 = 31$ shirts left.

**Answer:** There are **31** shirts left.


## Demo 3: Self-Consistency
Sample several reasoning chains (temperature > 0) and take the most common final answer.

In [5]:
def final_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None

prompt = (f"Q: {question}\nThink step by step, then end with 'The answer is <number>.'\nA:")
answers = []
for _ in range(5):
    out = generate(prompt, max_tokens=250, temperature=0.8)
    answers.append(final_number(out))

print("Votes :", Counter(answers))
print("Winner:", Counter(answers).most_common(1)[0][0], "(truth: 31)")

Votes : Counter({'31': 5})
Winner: 31 (truth: 31)


## Demo 4: RAG
Retrieve the most relevant document (TF-IDF here; real systems use embeddings + a vector DB) and add it to the prompt.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "Acme Corp's refund policy: items can be returned within 45 days with a receipt.",
    "Acme Corp support hours are 9am-5pm ICT, Monday to Friday.",
    "Acme Corp's Pro plan costs $29 per month and includes 5 seats.",
    "Acme Corp headquarters are located in Phnom Penh, Cambodia.",
]
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

def retrieve(query, k=1):
    scores = cosine_similarity(vec.transform([query]), doc_vecs).ravel()
    return [docs[i] for i in scores.argsort()[::-1][:k]]

def rag_answer(query, k=1):
    context = "\n".join(retrieve(query, k))
    return generate(f"Answer using only the context.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:")

q = "How many days do I have to return an item at Acme Corp?"
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q))

No context: I do not have access to specific return policies for Acme Corp. To find out how many days you have to return an item, you will need to check their official website, customer service page, or the packing slip/receipt for your purchase.
With RAG  : The context does not state how many days you have to return an item at Acme Corp.


---
# Hands-on Practice
Complete the `TODO`s. Re-run each cell and compare outputs. Small models are noisy, so try a few variations.

### Exercise 1: Few-shot extraction
Make the model turn a sentence into JSON with keys `name`, `city`, `age`. First try zero-shot, then add **2 examples** to get a clean, consistent format.

In [10]:
text = "Sokha, a 29-year-old engineer, just moved to Siem Reap."

# TODO: add two examples before the final input
zero_shot_prompt = f"""Extract name, city, and age as JSON.

Text: {text}
JSON:"""

few_shot_prompt = f"""Extract name, city, and age as JSON with keys "name", "city", "age".
Reply with the JSON object only.

Text: Mongkol, 16, works as a AI Engineer in Battambang.
JSON: {{"name": "Mongkol", "city": "Battambang", "age": 16}}

Text: After turning 22, Srey Neang moved to Phnom Penh for university.
JSON: {{"name": "Srey Neang", "city": "Phnom Penh", "age": 22}}

Text: {text}
JSON:"""

print("Zero-Shot: ", generate(zero_shot_prompt, max_tokens=60))
print("\nFew-Shot: ", generate(few_shot_prompt, max_tokens=60))

Zero-Shot:  ```json
{
  "name": "Sokha",
  "city": "Siem Reap",
  "age": 29
}
```

Few-Shot:  {"name": "Sokha", "city": "Siem Reap", "age": 29}


### Exercise 2: Chain-of-Thought
The puzzle below often trips up direct answers. Compare a **direct** prompt with a **CoT** prompt. Which is right? (Answer: 0.05, i.e. 5 cents)

In [13]:
puzzle = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much is the ball?"

direct = f"Q: {puzzle}\nGive only the final answer in dollars, nothing else.\nA:"

cot = (f"Q: {puzzle}\n"
       "Let's think step by step. Define a variable for the ball's price, "
       "write an equation, solve it, and check the result. "
       "Keep it short and use plain text (no LaTeX, no markdown). "
       "End with 'Answer: <number>' in dollars.\nA:")

print("Direct:", generate(direct, max_tokens=40))
print("CoT   :", generate(cot, max_tokens=500))

Direct: 50
CoT   : Let B be the cost of the bat.
Let L be the cost of the ball.

We are given that the total cost is $1.10:
B + L = 1.10

We are also given that the bat costs $1.00 more than the ball:
B = L + 1.00

Now substitute the second equation into the first equation:
(L + 1.00) + L = 1.10

Combine like terms:
2L + 1.00 = 1.10

Subtract 1.00 from both sides:
2L = 1.10 - 1.00
2L = 0.10

Divide by 2 to find L:
L = 0.10 / 2
L = 0.05

Check the result:
If the ball (L) costs $0.05, then the bat (B) costs $0.05 + $1.00 = $1.05.
The total cost is $1.05 + $0.05 = $1.10. This matches the given information.

Answer: 0.05


### Exercise 3: Self-Consistency
Write a function `self_consistent(prompt, k)` that samples `k` answers at `temperature=0.8` and returns the majority answer. Test it on the puzzle above with `k=1` vs `k=7`. Does voting help?

*Hint:* ask the model to end with `Answer: <number>` so extracting the answer is easy.

In [14]:
def self_consistent(prompt, k=5):
    answers = []
    for _ in range(k):
        out = generate(prompt, max_tokens=250, temperature=0.8)
        answers.append(final_number(out))

        print("Votes: ", Counter(answers))
        return Counter(answers).most_common(1)[0][0]

cot_prompt = f"Q: {puzzle}\nThink step by step, then end with 'Answer: <number>'.\nA:"

print(self_consistent(cot_prompt, k=1))
print(self_consistent(cot_prompt, k=7))

Votes:  Counter({'0.10': 1})
0.10
Votes:  Counter({'0': 1})
0


### Exercise 4: RAG
1. Add **two new facts** of your own to `docs` (re-fit the vectorizer after).
2. Ask a question that only your facts can answer.
3. Compare the answer with and without retrieval. What happens when the answer is **not** in the docs? Edit the RAG prompt so the model says *"I don't know"* in that case.

In [15]:
docs = docs + [
    "Acme Corp offers a 20% student discount on the Pro plan with a valid student ID of Kirirom Institute of Technology.",
    "Acme Corp's mobile app is available on Android and iOS and supports Khmer and English.",
]

vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

q = "Is there a student discount at Acme Corp?"

print("Retrieved :", retrieve(q))
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q))

q_missing = "Who is the CEO of Acme Corp?"
print("\nRetrieved :", retrieve(q_missing))
print("Old RAG   :", rag_answer(q_missing))

def rag_answer_strict(query, k=2):
    context = "\n".join(retrieve(query, k))
    prompt = f"""You are a support assistant for Acme Corp.
Answer the question using ONLY the context below.
If the context does not contain the answer, reply exactly: I don't know.
Do not use outside knowledge and do not guess.

Context:
{context}

Question: {query}
Answer:"""
    return generate(prompt, max_tokens=60)

print("\nStrict RAG (answer in docs)    :", rag_answer_strict(q))
print("Strict RAG (answer not in docs):", rag_answer_strict(q_missing))

Retrieved : ['Acme Corp offers a 20% student discount on the Pro plan with a valid student ID of Kirirom Institute of Technology.']
No context: I do not have access to real-time, specific information about the policies or discounts offered by a particular company like Acme Corp.

To find out if there is a student discount at Acme Corp, you should check their official website, contact their customer service, or look for a "Pricing," "
With RAG  : Yes, Acme Corp offers a 20% student discount on the Pro plan with a valid student ID of Kirirom Institute of Technology.

Retrieved : ['Acme Corp offers a 20% student discount on the Pro plan with a valid student ID of Kirirom Institute of Technology.']
Old RAG   : Answer: The context does not mention who the CEO of Acme Corp is.

Strict RAG (answer in docs)    : Acme Corp offers a 20% student discount on the Pro plan with a valid student ID of Kirirom Institute of Technology.
Strict RAG (answer not in docs): I don't know.


### Challenge (optional)
Pick a task from your own work (e.g. classify support tickets, summarize, translate to Khmer). Design a prompt, then improve it by adding **a role, a format constraint, and examples**. Note which change helped most.

---
## Recap
- **Few-shot**: examples fix format and style
- **CoT**: reasoning steps help multi-step problems
- **Self-consistency**: voting across samples improves reliability, at k× the cost
- **RAG**: grounds answers in your own data

Next: when prompting is not enough, **fine-tuning** updates the weights.

In [16]:
sentences = [
    "The exam will start at 8 a.m. tomorrow.",
    "Students must bring their ID card.",
    "Machine learning is a branch of artificial intelligence.",
    "Please restart your computer before the class.",
]

references = [
    "ការប្រឡងនឹងចាប់ផ្តើមនៅម៉ោង ៨ ព្រឹកស្អែក។",
    "សិស្សត្រូវយកកាតសម្គាល់ខ្លួនមកជាមួយ។",
    "ការរៀនរបស់ម៉ាស៊ីន (Machine Learning) គឺជាផ្នែកមួយនៃបញ្ញាសិប្បនិម្មិត។",
    "សូមចាប់ផ្តើមកុំព្យូទ័ររបស់អ្នកឡើងវិញមុនពេលចូលរៀន។",
]

def v0(s):
    return f"Translate to Khmer: {s}"

def v1(s):
    return f"""Translate the English sentence into Khmer.
Reply with the Khmer translation only. No explanation, no romanization, no English.

English: {s}
Khmer:"""

def v2(s):
    return "You are a professional English-Khmer translator for a Cambodian university.\n" + v1(s)

def v3(s):
    return f"""You are a professional English-Khmer translator for a Cambodian university.
Translate the English sentence into Khmer.
Reply with the Khmer translation only. No explanation, no romanization, no English.
Keep technical terms in English in brackets after the Khmer term.

English: Good morning, class.
Khmer: អរុណសួស្តី សិស្សទាំងអស់គ្នា។

English: Please submit your homework by Friday.
Khmer: សូមប្រគល់កិច្ចការផ្ទះរបស់អ្នកមុនថ្ងៃសុក្រ។

English: The library is closed on Sunday.
Khmer: បណ្ណាល័យបិទនៅថ្ងៃអាទិត្យ។

English: {s}
Khmer:"""

def khmer_ratio(text):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0
    khmer = [c for c in letters if "\u1780" <= c <= "\u17FF"]
    return round(100 * len(khmer) / len(letters))

for name, fn in [("v0 bare", v0), ("v1 +format", v1), ("v2 +role", v2), ("v3 +examples", v3)]:
    print(f"\n===== {name} =====")
    for s, ref in zip(sentences, references):
        out = generate(fn(s), max_tokens=150)
        print(f"EN : {s}")
        print(f"OUT: {out}   [{khmer_ratio(out)}% Khmer]")
        print(f"REF: {ref}\n")


===== v0 bare =====
EN : The exam will start at 8 a.m. tomorrow.
OUT: Here are a few ways to translate "The exam will start at 8 a.m. tomorrow" into Khmer, depending on the desired level of formality:

**Most Common and Natural Translation:**

> **ការប្រឡងនឹងចាប់ផ្តើមនៅម៉ោង ៨ព្រឹកថ្ងៃស្អែក។**
> (Kā prâlŏng nŭng chăp phdaem nŭng māong pĕt prĕk thngai sâek.)

**Breakdown:**

* **ការប្រឡង (Kā prâlŏng):** The exam (or test)
* **នឹង (nŭng):** Will (indicates future action)
* **ចាប់   [12% Khmer]
REF: ការប្រឡងនឹងចាប់ផ្តើមនៅម៉ោង ៨ ព្រឹកស្អែក។

EN : Students must bring their ID card.
OUT: Here are a few ways to translate "Students must bring their ID card" into Khmer, depending on the desired level of formality:

**Most Common and Natural Translation (Slightly formal/instructional):**

> **សិស្សត្រូវយកប័ណ្ណសម្គាល់ខ្លួនមក។**
> *(*Sĕs trŏu yôk bân sâmkŏal khluŏn môk.)*

* **សិស្ស (Sĕs):** Students
* **ត្រូវ (trŏu):** Must/Have to
* **យក (yôk):** Bring/Take
* **ប័ណ្ណសម្គាល់ខ្លួន (bân sâmkŏal kh 